In [1]:
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.


In [2]:
!pip install speechrecognition gtts transformers torch

INFO: pip is looking at multiple versions of huggingface-hub to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of typer to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 32.9/32.9 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 668.2/668.2 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.5/122.5 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.4.1
    Uninstalling click-8.4.1:
      Successfully uninstalled click-8.4.1
  Attempting uninstall: typer
    Found existing installation: typer 0.25.1
    Uninstalling typer-0.25.1:
      Successfully uninstalled typer-0.25.1
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub

In [3]:
import os
import torch
import tempfile
import wave
import speech_recognition as sr

from transformers import AutoTokenizer, AutoModelForCausalLM
from gtts import gTTS
from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode


In [4]:
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-medium")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-medium")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/863M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/863M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [5]:
raw_audio_file = "user_audio.webm"

def save_audio(b64):
    audio_bytes = b64decode(b64)
    with open(raw_audio_file, "wb") as f:
        f.write(audio_bytes)
    print("✅ Audio saved:", raw_audio_file)

output.register_callback("notebook.save_audio", save_audio)


In [7]:
RECORD = """
const sleep = time => new Promise(resolve => setTimeout(resolve, time));

const record = async () => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true });
  const recorder = new MediaRecorder(stream);
  recorder.start();
  await sleep(5000);
  recorder.stop();

  return await new Promise(resolve => {
    recorder.ondataavailable = e => resolve(e.data);
  });
};

record().then(blob => {
  const reader = new FileReader();
  reader.readAsDataURL(blob);
  reader.onloadend = () => {
    const base64data = reader.result.split(',')[1];
    google.colab.kernel.invokeFunction(
      'notebook.save_audio',
      [base64data],
      {}
    );
  };
});
"""

display(Javascript(RECORD))


<IPython.core.display.Javascript object>

✅ Audio saved: user_audio.webm


In [8]:
!ffmpeg -y -i user_audio.webm -ac 1 -ar 16000 user_audio.wav

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [9]:
with wave.open("user_audio.wav", "rb") as wf:
    print("Channels:", wf.getnchannels())
    print("Sample Rate:", wf.getframerate())
    print("Frames:", wf.getnframes())


Channels: 1
Sample Rate: 16000
Frames: 72000


In [10]:
recognizer = sr.Recognizer()

with sr.AudioFile("user_audio.wav") as source:
    audio_data = recognizer.record(source)

try:
    user_text = recognizer.recognize_google(audio_data)
    print("🧑 You said:", user_text)
except sr.UnknownValueError:
    print("❌ Could not understand audio")
    user_text = ""
except sr.RequestError as e:
    print("❌ Speech service error:", e)
    user_text = ""


🧑 You said: Hi how are you


In [11]:
def chatbot_response(user_input):
    input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"
    )

    output_ids = model.generate(
        input_ids,
        max_length=120,
        pad_token_id=tokenizer.eos_token_id
    )

    reply = tokenizer.decode(
        output_ids[:, input_ids.shape[-1]:][0],
        skip_special_tokens=True
    )

    return reply


In [12]:
if user_text:
    bot_reply = chatbot_response(user_text)
else:
    bot_reply = "Sorry, I couldn't hear you."

print("🤖 Siri-like Bot:", bot_reply)


[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


🤖 Siri-like Bot: I'm good , how are you ?


In [13]:
tts = gTTS(text=bot_reply, lang="en")

temp_audio = tempfile.NamedTemporaryFile(delete=False, suffix=".mp3")
tts.save(temp_audio.name)

display(Audio(temp_audio.name, autoplay=True))
